[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harold-Ohandja/Moderna-Quantum-RNA-/blob/main/notebooks/05_Noise_Robustness.ipynb)


## Setup — run this cell first

Safe to run whether you're in **Google Colab** or already working locally inside
a clone of this repo:

- **On Colab**: installs the missing packages (ViennaRNA + Qiskit stack), does a
  clean clone of the repo (removing any stale partial clone first, so re-running
  this cell is always safe), and moves into `notebooks/`.
- **Running locally**, with the repo already cloned and this notebook opened from
  its actual `notebooks/` folder: detected automatically, and this cell just
  confirms the working directory instead of re-cloning anything.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Colab: installing dependencies and cloning the repo...")
    !pip install -q "viennarna>=2.7.0" "qiskit>=2.5.0" "qiskit-algorithms>=0.4.0" "qiskit-aer>=0.17.0"
    %cd /content
    !rm -rf Moderna-Quantum-RNA-
    !git clone -q https://github.com/Harold-Ohandja/Moderna-Quantum-RNA-.git
    %cd /content/Moderna-Quantum-RNA-/notebooks
    print("Done. Working directory:", os.getcwd())
else:
    print("Not on Colab, assuming the repo is already cloned locally.")
    print("Working directory:", os.getcwd())

# 05 — Noise Robustness (Optional Advanced Task)

**WISER Quantum+AI 2026 — Moderna Challenge**

This notebook addresses the optional task of testing how the quantum result holds
up under noise. It takes the one case the project already solves exactly —
`GCGCAUACGC`, 7 qubits, where CVaR-VQE reproduces the ViennaRNA MFE structure with
a 0.0 kcal/mol energy gap (notebook 04) — and asks what happens to that result when
the sampling is no longer ideal.

Three regimes, all on the **same** solver, ansatz and QUBO as notebook 04:

1. **Noiseless baseline** — 1024 shots, no noise model. The control.
2. **Sampling noise only** — no noise model, but the shot budget is varied
   (128 / 512 / 2048). Isolates finite-shot estimator noise.
3. **Depolarizing noise** — 1024 shots plus a hardware-inspired Aer `NoiseModel`:
   depolarizing error on the single-qubit gates, and 10x that rate on the
   two-qubit entangling gates, which is the usual ratio on real devices.

**Scope and honesty notes:**

- This is **local Aer simulation only**. No real quantum hardware, no IBM Quantum
  account, no cloud backend is used or required.
- The depolarizing model is *hardware-inspired, not calibrated to any specific
  device*. It shows a trend under increasing error rates; it does not predict what
  any particular backend would return.
- Every number below is produced by actually running the solver in this notebook
  and is read out of the `noise_df` DataFrame. Nothing is pasted in.
- Each configuration is repeated over several seeds and **all** of them are
  reported, including any that fail, so the variability is visible rather than
  hidden behind a best-of.

> **Runtime note:** the sweep in section 2 runs 35 solves and takes roughly 2 minutes; if you are executing all the notebooks in one batch, run this one standalone.

In [ ]:
import sys, time
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import RNA
from qiskit_aer.primitives import SamplerV2
from qiskit_aer.noise import NoiseModel, depolarizing_error

from classical.evaluate_energy import evaluate_structure_energy, calculate_energy_gap
from quantum.qubo import build_qubo_matrix
from quantum.vqe_solver import solve_cvar_vqe

SEQUENCE = "GCGCAUACGC"          # smallest case, and the one notebook 04 solves exactly
ALPHA = 0.1                       # same CVaR quantile as the primary solver
SEEDS = [11, 22, 33, 44, 55]      # repetitions, all reported

qubo, pairs = build_qubo_matrix(SEQUENCE)
REF_STRUCT, REF_MFE = RNA.fold(SEQUENCE)

print(f"Sequence      : {SEQUENCE} ({len(SEQUENCE)} nt, {len(pairs)} qubits)")
print(f"ViennaRNA MFE : {REF_STRUCT}  ({REF_MFE:.2f} kcal/mol)")
print(f"Seeds         : {SEEDS}")

## 1. The noise model

The ansatz transpiles to `ry` (single-qubit) and `cz` (two-qubit) gates, so the
depolarizing error is attached to exactly those. The two-qubit rate is 10x the
single-qubit rate.

A `FakeBackend` from `qiskit-ibm-runtime` would be the other option, but that
package is not a dependency of this project and pulling it in would add install
weight on Colab for every judge, to get a model calibrated to a device we are not
claiming to target. A small explicit model keeps the notebook self-contained and
makes the assumption visible instead of hidden inside a snapshot.

In [ ]:
def depolarizing_noise_model(p1: float, ratio: float = 10.0) -> NoiseModel:
    """Hardware-inspired depolarizing model: `p1` on single-qubit gates,
    `ratio * p1` on two-qubit gates (capped at 1.0)."""
    model = NoiseModel()
    model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["ry"])
    model.add_all_qubit_quantum_error(depolarizing_error(min(ratio * p1, 1.0), 2), ["cz"])
    return model


# One row per configuration; the run loop below expands these over SEEDS.
CONFIGS = [dict(arm="noiseless", label="noiseless", shots=1024, p1=None)]
CONFIGS += [dict(arm="shot noise", label=f"{s} shots", shots=s, p1=None)
            for s in (128, 512, 2048)]
CONFIGS += [dict(arm="depolarizing", label=f"depol p={p}", shots=1024, p1=p)
            for p in (0.001, 0.005, 0.02)]

pd.DataFrame(CONFIGS)

## 2. Run every configuration over every seed

`solve_cvar_vqe` takes an optional pre-built `sampler`. That is the injection point
used here: shot count and noise model are both properties of the sampler, because
`SamplingVQE` calls `sampler.run()` without a shots argument. The solver, ansatz,
CVaR quantile and QUBO are otherwise untouched — identical to notebook 04.

In [ ]:
rows = []
for cfg in CONFIGS:
    for seed in SEEDS:
        options = None
        if cfg["p1"] is not None:
            options = {"backend_options": {"noise_model": depolarizing_noise_model(cfg["p1"])}}
        sampler = SamplerV2(default_shots=cfg["shots"], seed=seed, options=options)

        t0 = time.perf_counter()
        out = solve_cvar_vqe(qubo, pairs, len(SEQUENCE), alpha=ALPHA,
                             seed=seed, sampler=sampler)
        runtime = time.perf_counter() - t0

        structure = out["structure"]
        energy = evaluate_structure_energy(SEQUENCE, structure)
        gap = calculate_energy_gap(energy, REF_MFE)

        rows.append({
            "arm": cfg["arm"], "config": cfg["label"], "shots": cfg["shots"],
            "p1": cfg["p1"], "seed": seed, "structure": structure,
            "energy_kcal": energy, "match": structure == REF_STRUCT,
            "gap_kcal": gap["absolute_gap_kcal"], "runtime_sec": round(runtime, 3),
        })
    done = [r for r in rows if r["config"] == cfg["label"]]
    print(f"{cfg['label']:>12s}: {sum(r['match'] for r in done)}/{len(SEEDS)} recovered the MFE")

noise_df = pd.DataFrame(rows)
print(f"\nTotal runs: {len(noise_df)}  |  total solver time: {noise_df['runtime_sec'].sum():.1f}s")
noise_df

## 3. Summary

`success` counts how many seeds recovered ViennaRNA's exact MFE structure. The gap
columns are the min / median / max energy gap **across seeds**, so a spread of zero
means every repetition agreed.

In [ ]:
summary = (
    noise_df.groupby(["arm", "config"], sort=False)
    .agg(success=("match", "sum"),
         runs=("match", "size"),
         gap_min=("gap_kcal", "min"),
         gap_median=("gap_kcal", "median"),
         gap_max=("gap_kcal", "max"),
         mean_runtime_sec=("runtime_sec", "mean"))
    .reset_index()
)
summary["success"] = summary["success"].astype(int).astype(str) + "/" + summary["runs"].astype(str)
summary = summary.drop(columns="runs")
summary["mean_runtime_sec"] = summary["mean_runtime_sec"].round(2)
summary

## 4. Energy gap by configuration

One point per seed, so any spread within a configuration is visible directly. The
dashed line at 0.0 is the ViennaRNA MFE — a point on that line is an exact
structural match.

In [ ]:
ARM_COLORS = {"noiseless": "#4C72B0", "shot noise": "#DD8452", "depolarizing": "#C44E52"}

order = list(dict.fromkeys(noise_df["config"]))

# Fix the y-range up front from the data. When every gap is 0.0 the autoscaled
# range is degenerate, so fall back to a fixed window that still shows the
# 0.0 reference line clearly.
gap_max = float(noise_df["gap_kcal"].max())
top = gap_max * 1.35 if gap_max > 0 else 1.0
bottom = -0.12 * top

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.set_ylim(bottom, top)

for x, config in enumerate(order):
    sub = noise_df[noise_df["config"] == config]
    arm = sub["arm"].iloc[0]
    jitter = np.linspace(-0.13, 0.13, len(sub))
    ax.scatter(x + jitter, sub["gap_kcal"], s=70, color=ARM_COLORS[arm],
               edgecolor="white", linewidth=0.8, zorder=3)
    ax.text(x, top * 0.93, f"{int(sub['match'].sum())}/{len(sub)}", ha="center",
            va="center", fontsize=8.5, color="#444444")

ax.axhline(0.0, color="#888888", linestyle="--", linewidth=1.2, zorder=1)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=20, ha="right")
ax.set_ylabel("Energy gap vs. ViennaRNA MFE (kcal/mol)")
ax.set_title(f"CVaR-VQE on {SEQUENCE} under noise "
             f"(one point per seed; n/{len(SEEDS)} = seeds recovering the exact MFE)",
             fontsize=11)

if gap_max == 0:
    ax.text((len(order) - 1) / 2, top * 0.45,
            "every run landed exactly on the ViennaRNA MFE (gap 0.0)",
            ha="center", va="center", fontsize=9, style="italic", color="#777777")

handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=c, label=a,
                      markeredgecolor="white") for a, c in ARM_COLORS.items()]
ax.legend(handles=handles, loc="center left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 5. What this shows

The cells above are the evidence; this section is written to be read against the
table in section 3 rather than in place of it.

Two things worth stating plainly:

- **A negative result is still a result.** If this instance holds its exact MFE
  across the whole range tested, the honest conclusion is "this problem is robust
  over the error rates simulated here", not that the approach is noise-proof in
  general. The error rates were fixed in advance and were *not* widened to
  manufacture a degradation.
- **This is one 7-qubit instance.** `GCGCAUACGC` has 7 candidate pairs and a
  depth-11 ansatz, which is small enough that shot noise averages out and
  depolarizing error has few gates to accumulate on. Larger sequences have both
  more qubits and deeper circuits, so robustness here should not be extrapolated
  to the 87-qubit case in notebook 04. Establishing where it breaks down would
  need the larger instances that are already beyond local simulation — which is
  the same practical wall the scaling analysis in Milestone 6 identifies.